# Network Rail SCHEDULE Data Structure

We are ingesting SCHEDULE data from Network Rail's [Open Data Feed](https://publicdatafeeds.networkrail.co.uk/ntrod/). This notebook explores the structure of the data so we can use it for analytics.

Documentation is available online in a [wiki](https://wiki.openraildata.com/index.php?title=SCHEDULE). It is impressively comprehensive - our work here is to simply confirm the data is shaped as described there.

## Ingested data

We ingest gzipped NDJSON data from Network Rail's SCHEDULE endpoint. According to the wiki, each (full snapshot) file consists of the following objects:
- a single header record
- zero or more TIPLOC (timing point location) record
- zero or more association records
- zero or more schedule records
- an EOF record

Due to the NDJSON format, each record comprises one line of the uncompressed file.

More details can be found [on the wiki](https://wiki.openraildata.com/index.php/JSON_File_Format).

## What we'll examine

We need to take a look at a few examples of each record type. The header row is well understood as it formed a crucial source of metadata for the ingest process.

In [ ]:
from google.cloud import storage
import gzip
import json
import io

BUCKET_NAME = "railway-analytics-508615-network-rail-open-data-raw"
OBJECT_NAME = "schedules/full/sequence=5251/schedules.json.gz"

In [2]:
def iter_schedule_records(
    bucket_name: str,
    object_name: str,
):
    client = storage.Client()
    blob = client.bucket(bucket_name).blob(object_name)

    with blob.open("rb") as raw:
        with gzip.GzipFile(fileobj=raw) as gz:
            with io.TextIOWrapper(gz, encoding="utf-8") as text:
                for line in text:
                    yield json.loads(line)

In [3]:
from collections import defaultdict

wanted = {
    "JsonTimetableV1": 1,
    "TiplocV1": 3,
    "JsonAssociationV1": 3,
    "JsonScheduleV1": 10,
}

samples = defaultdict(list)

line_number = 0
for line_number, record in enumerate(
    iter_schedule_records(BUCKET_NAME, OBJECT_NAME),
    start=1,
):
    record_type = next(iter(record))

    if (
        record_type in wanted
        and len(samples[record_type]) < wanted[record_type]
    ):
        samples[record_type].append(record[record_type])

    if all(
        len(samples[record_type]) >= count
        for record_type, count in wanted.items()
    ):
        break

line_number

95406

In [4]:
{
    record_type: len(records)
    for record_type, records in samples.items()
}

{'JsonTimetableV1': 1,
 'TiplocV1': 3,
 'JsonAssociationV1': 3,
 'JsonScheduleV1': 10}

## `JsonTimetableV1`: the header row

In [46]:
from pprint import pprint

header = samples["JsonTimetableV1"][0]

print("Header:")
pprint(header)

print("\nMetadata:")
pprint(header["Metadata"])

print("\nTop-level key set:")
print(frozenset(header.keys()))

print("\nMetadata key set:")
print(frozenset(header["Metadata"].keys()))

Header:
{'Metadata': {'sequence': 5251, 'type': 'full'},
 'Sender': {'application': 'NTROD',
            'component': 'SCHEDULE',
            'organisation': 'Rockshore'},
 'classification': 'public',
 'owner': 'Network Rail',
 'timestamp': 1791246721}

Metadata:
{'sequence': 5251, 'type': 'full'}

Top-level key set:
frozenset({'classification', 'Metadata', 'owner', 'Sender', 'timestamp'})

Metadata key set:
frozenset({'type', 'sequence'})


## TIPLOC: timing point location codes

These entries describe locations that can appear in schedules. In a full snapshot file, all location records have a `transaction_type` of `Create`, but in update files, this can be `Delete` or `Update` according to the [Wiki](https://wiki.openraildata.com/index.php/Tiploc_Records).

In [5]:
from pprint import pprint

for tiploc in samples["TiplocV1"]:
    pprint(tiploc)
    print()

{'crs_code': None,
 'description': None,
 'nalco': '081601',
 'stanox': '00005',
 'tiploc_code': 'AACHEN',
 'tps_description': 'AACHEN',
 'transaction_type': 'Create'}

{'crs_code': 'AER',
 'description': 'SQWAR ALBAN SQ',
 'nalco': '027400',
 'stanox': None,
 'tiploc_code': 'ABARASQ',
 'tps_description': 'ABERAERON ALBAN SQUARE',
 'transaction_type': 'Create'}

{'crs_code': None,
 'description': None,
 'nalco': '385964',
 'stanox': '78128',
 'tiploc_code': 'ABCWM',
 'tps_description': 'ABERCWMBOI',
 'transaction_type': 'Create'}



The schema seems stable. We will confirm this and compute null counts next.

In [6]:
from collections import Counter

expected_keys = {
    "crs_code",
    "description",
    "nalco",
    "stanox",
    "tiploc_code",
    "tps_description",
    "transaction_type",
}

tiploc_count = 0
key_sets = Counter()
null_counts = Counter()
transaction_types = Counter()

seen_tiploc = False

for record in iter_schedule_records(BUCKET_NAME, OBJECT_NAME):
    record_type = next(iter(record))

    if record_type == "TiplocV1":
        seen_tiploc = True
        tiploc = record["TiplocV1"]

        tiploc_count += 1
        key_sets[frozenset(tiploc.keys())] += 1

        for key, value in tiploc.items():
            if value is None:
                null_counts[key] += 1

        transaction_types[tiploc["transaction_type"]] += 1

    elif seen_tiploc:
        break

In [9]:
print(f"Total TIPLOC records seen: {tiploc_count}")
print("\nKey sets:")
pprint(key_sets)
print("\nNull counts:")
pprint(null_counts)
print("\nTransaction types (full snapshot):")
pprint(transaction_types)

Total TIPLOC records seen: 12096

Key sets:
Counter({frozenset({'stanox', 'tps_description', 'crs_code', 'tiploc_code', 'nalco', 'description', 'transaction_type'}): 12096})

Null counts:
Counter({'description': 9269, 'crs_code': 8520, 'stanox': 940})

Transaction types (full snapshot):
Counter({'Create': 12096})


Indeed, the schemas match perfectly over 12.1k objects. Four of the fields are never null in this snapshot:
- `tiploc_code`, the natural record identifier
- `transaction_type`
- `nalco`, national location code
- `tps_description`

`stanox` is occasionally null for non-station locations, as is to be expected. `description` and `crs_code` (CRS = 3-alpha code) are more often null, about 75% and 70% of the time respectively.

## Association: relationships between scheduled services

From the [Wiki page](https://wiki.openraildata.com/index.php/Association_Records), these are flat records encoding details of the relationship between two scheduled services (often called "trains" in documentation). There are a fair few codes used here, so it's worth exploring them a little and ensuring we convert them correctly to a suitable database value.

In [ ]:
for association in samples["JsonAssociationV1"]:
    pprint(association)
    print()

{'CIF_stp_indicator': 'P',
 'assoc_days': '0000001',
 'assoc_end_date': '2026-12-06T00:00:00Z',
 'assoc_location_suffix': None,
 'assoc_start_date': '2026-05-17T00:00:00Z',
 'assoc_train_uid': 'Y14938',
 'base_location_suffix': None,
 'category': 'NP',
 'date_indicator': 'S',
 'diagram_type': 'T',
 'location': 'KNGX',
 'main_train_uid': 'C00202',
 'transaction_type': 'Create'}

{'CIF_stp_indicator': 'P',
 'assoc_days': '0000001',
 'assoc_end_date': '2026-12-06T00:00:00Z',
 'assoc_location_suffix': None,
 'assoc_start_date': '2026-05-17T00:00:00Z',
 'assoc_train_uid': 'Y14906',
 'base_location_suffix': None,
 'category': 'NP',
 'date_indicator': 'S',
 'diagram_type': 'T',
 'location': 'BRADIN',
 'main_train_uid': 'C00203',
 'transaction_type': 'Create'}

{'CIF_stp_indicator': 'P',
 'assoc_days': '0000001',
 'assoc_end_date': '2026-12-06T00:00:00Z',
 'assoc_location_suffix': None,
 'assoc_start_date': '2026-05-17T00:00:00Z',
 'assoc_train_uid': 'Y14837',
 'base_location_suffix': None,
 '

In [18]:
association_count = 0
association_key_sets = Counter()
association_null_counts = Counter()

seen_association = False

for record in iter_schedule_records(BUCKET_NAME, OBJECT_NAME):
    record_type = next(iter(record))

    if record_type == "JsonAssociationV1":
        seen_association = True
        association = record["JsonAssociationV1"]

        association_count += 1
        association_key_sets[frozenset(association.keys())] += 1

        for key, value in association.items():
            if value is None:
                association_null_counts[key] += 1

    elif seen_association:
        break


print(f"Total association records seen: {association_count}")
print("\nKey sets:")
pprint(association_key_sets)
print("\nNull counts:")
pprint(association_null_counts)

Total association records seen: 83299

Key sets:
Counter({frozenset({'date_indicator', 'location', 'assoc_location_suffix', 'assoc_train_uid', 'CIF_stp_indicator', 'main_train_uid', 'assoc_days', 'diagram_type', 'category', 'assoc_end_date', 'assoc_start_date', 'transaction_type', 'base_location_suffix'}): 83299})

Null counts:
Counter({'assoc_location_suffix': 83178, 'base_location_suffix': 80530})


Again, the schema consistency is faultless. Almost all of the fields are non-null, the exceptions being two `_suffix` fields which are only used when a train meets the same TIPLOC at multiple points on the route. Some of the fields are coded but this is an issue for staging rather than the raw landing layer. The Wiki documentation is excellent on these codes.

## Schedules

Finally we get to the meat of the file: the actual schedule records. These have a more deeply nested structure in comparison to the other two record types, which are fundamentally flat.

In [19]:
schedule = samples["JsonScheduleV1"][0]

print("Schedule-level fields:")
pprint({
    key: value
    for key, value in schedule.items()
    if key != "schedule_segment"
})

segment = schedule["schedule_segment"]

print("\nSchedule segment fields:")
pprint({
    key: value
    for key, value in segment.items()
    if key != "schedule_location"
})

locations = segment["schedule_location"]

print(f"\nNumber of locations: {len(locations)}")

print("\nFirst 5 locations:")
for location in locations[:5]:
    pprint(location)
    print()

Schedule-level fields:
{'CIF_bank_holiday_running': None,
 'CIF_stp_indicator': 'P',
 'CIF_train_uid': 'C00038',
 'applicable_timetable': 'Y',
 'atoc_code': 'NT',
 'new_schedule_segment': {'traction_class': '', 'uic_code': ''},
 'schedule_days_runs': '0000001',
 'schedule_end_date': '2026-11-08',
 'schedule_start_date': '2026-05-17',
 'train_status': 'P',
 'transaction_type': 'Create'}

Schedule segment fields:
{'CIF_business_sector': '??',
 'CIF_catering_code': None,
 'CIF_connection_indicator': None,
 'CIF_course_indicator': 1,
 'CIF_headcode': '',
 'CIF_operating_characteristics': None,
 'CIF_power_type': 'DMU',
 'CIF_reservations': None,
 'CIF_service_branding': '',
 'CIF_sleepers': None,
 'CIF_speed': '075',
 'CIF_timing_load': 'S',
 'CIF_train_category': 'EE',
 'CIF_train_class': None,
 'CIF_train_service_code': '22974000',
 'signalling_id': '5H63'}

Number of locations: 3

First 5 locations:
{'departure': '0905',
 'engineering_allowance': None,
 'line': None,
 'location_type': '

In [22]:
schedule_count = 0
location_count = 0

schedule_key_sets = Counter()
segment_key_sets = Counter()
new_segment_key_sets = Counter()
location_key_sets = Counter()

schedule_null_counts = Counter()
segment_null_counts = Counter()
new_segment_null_counts = Counter()
location_null_counts = Counter()

new_segment_missing_count = 0
schedule_location_missing_count = 0

for record in iter_schedule_records(BUCKET_NAME, OBJECT_NAME):
    if "JsonScheduleV1" not in record:
        continue

    schedule = record["JsonScheduleV1"]
    schedule_count += 1

    schedule_key_sets[frozenset(schedule.keys())] += 1

    for key, value in schedule.items():
        if key not in {"schedule_segment", "new_schedule_segment"} and value is None:
            schedule_null_counts[key] += 1

    segment = schedule["schedule_segment"]
    segment_key_sets[frozenset(segment.keys())] += 1

    for key, value in segment.items():
        if key != "schedule_location" and value is None:
            segment_null_counts[key] += 1

    locations = segment.get("schedule_location")

    if locations is None:
        schedule_location_missing_count += 1
    else:
        for location in locations:
            location_count += 1
            location_key_sets[frozenset(location.keys())] += 1

            for key, value in location.items():
                if value is None:
                    location_null_counts[key] += 1

    new_segment = schedule.get("new_schedule_segment")

    if new_segment is None:
        new_segment_missing_count += 1
    else:
        new_segment_key_sets[frozenset(new_segment.keys())] += 1

        for key, value in new_segment.items():
            if value is None:
                new_segment_null_counts[key] += 1

In [23]:
print("Schedules:", schedule_count)
print("Locations:", location_count)

print("\nSchedule key sets:")
print(schedule_key_sets)

print("\nSegment key sets:")
print(segment_key_sets)

print("\nSchedules missing schedule_location:")
print(schedule_location_missing_count)

print("\nNew segment missing:")
print(new_segment_missing_count)

print("\nNew segment key sets:")
print(new_segment_key_sets)

print("\nLocation key sets:")
print(location_key_sets)

print("\nSchedule nulls:")
print(schedule_null_counts)

print("\nSegment nulls:")
print(segment_null_counts)

print("\nNew segment nulls:")
print(new_segment_null_counts)

print("\nLocation nulls:")
print(location_null_counts)

Schedules: 728364
Locations: 10145025

Schedule key sets:
Counter({frozenset({'atoc_code', 'schedule_end_date', 'CIF_stp_indicator', 'CIF_train_uid', 'CIF_bank_holiday_running', 'new_schedule_segment', 'schedule_start_date', 'applicable_timetable', 'schedule_days_runs', 'transaction_type', 'schedule_segment', 'train_status'}): 587536, frozenset({'schedule_end_date', 'CIF_stp_indicator', 'CIF_train_uid', 'CIF_bank_holiday_running', 'schedule_start_date', 'schedule_days_runs', 'transaction_type', 'schedule_segment', 'train_status'}): 140828})

Segment key sets:
Counter({frozenset({'signalling_id', 'CIF_business_sector', 'CIF_timing_load', 'CIF_train_class', 'CIF_reservations', 'CIF_headcode', 'CIF_speed', 'CIF_power_type', 'CIF_sleepers', 'CIF_course_indicator', 'CIF_catering_code', 'CIF_train_category', 'CIF_service_branding', 'CIF_connection_indicator', 'CIF_train_service_code', 'schedule_location', 'CIF_operating_characteristics'}): 587536, frozenset({'signalling_id', 'CIF_business_se

In [ ]:
from collections import Counter

schedule_shapes_by_transaction = Counter()

for record in iter_schedule_records(BUCKET_NAME, OBJECT_NAME):
    if "JsonScheduleV1" not in record:
        continue

    schedule = record["JsonScheduleV1"]

    has_locations = (
        "schedule_location"
        in schedule["schedule_segment"]
    )

    schedule_shapes_by_transaction[
        (
            schedule["transaction_type"],
            has_locations,
            "new_schedule_segment" in schedule,
        )
    ] += 1

schedule_shapes_by_transaction

Counter({('Create', True, True): 587536, ('Create', False, False): 140828})

This shows two clear variants of schedule records: standard schedules and STP (short term plan) cancellations. The empty-ish records are cancellations which simply mark a schedule as non-running for the given period of time.

There are also three clear variants of schedule locations: origin, intermediate, and terminus.

While these have slightly different shapes, within each variant the schema is rigid. In the raw landing layer we will rely on `NULL`s to represent unfilled fields but we will **aggressively assert schema shape in the staging layer**.

## Update files

We must also inspect the update files. While full snapshots only contain `Create` records, update files may include `Delete` rows or (depending on the record type) `Update` rows.

In [25]:
UPDATE_OBJECTS = {
    5251: "schedules/update/sequence=5251/schedules.json.gz",
    5252: "schedules/update/sequence=5252/schedules.json.gz",
    5253: "schedules/update/sequence=5253/schedules.json.gz",
}

In [26]:
record_type_counts = Counter()
transaction_type_counts = Counter()
key_sets = defaultdict(Counter)

for sequence, object_name in UPDATE_OBJECTS.items():
    for record in iter_schedule_records(BUCKET_NAME, object_name):
        record_type = next(iter(record))
        payload = record[record_type]

        record_type_counts[(sequence, record_type)] += 1

        transaction_type = (
            payload.get("transaction_type")
            if isinstance(payload, dict)
            else None
        )

        if transaction_type is not None:
            transaction_type_counts[
                (sequence, record_type, transaction_type)
            ] += 1

        if isinstance(payload, dict):
            key_sets[
                (sequence, record_type, transaction_type)
            ][frozenset(payload.keys())] += 1

In [27]:
print("Record type counts:")
for key, count in sorted(record_type_counts.items()):
    print(key, count)

print("\nTransaction type counts:")
for key, count in sorted(transaction_type_counts.items()):
    print(key, count)

for group, counts in sorted(key_sets.items(), key=str):
    print("\n", group)

    for keys, count in counts.most_common():
        print(f"  {count:>8}  {sorted(keys)}")

Record type counts:
(5251, 'EOF') 1
(5251, 'JsonAssociationV1') 1134
(5251, 'JsonScheduleV1') 11585
(5251, 'JsonTimetableV1') 1
(5252, 'EOF') 1
(5252, 'JsonAssociationV1') 1352
(5252, 'JsonScheduleV1') 7040
(5252, 'JsonTimetableV1') 1
(5252, 'TiplocV1') 5
(5253, 'EOF') 1
(5253, 'JsonAssociationV1') 1128
(5253, 'JsonScheduleV1') 7563
(5253, 'JsonTimetableV1') 1

Transaction type counts:
(5251, 'JsonAssociationV1', 'Create') 987
(5251, 'JsonAssociationV1', 'Delete') 147
(5251, 'JsonScheduleV1', 'Create') 9633
(5251, 'JsonScheduleV1', 'Delete') 1952
(5252, 'JsonAssociationV1', 'Create') 1167
(5252, 'JsonAssociationV1', 'Delete') 185
(5252, 'JsonScheduleV1', 'Create') 5204
(5252, 'JsonScheduleV1', 'Delete') 1836
(5252, 'TiplocV1', 'Create') 5
(5253, 'JsonAssociationV1', 'Create') 920
(5253, 'JsonAssociationV1', 'Delete') 208
(5253, 'JsonScheduleV1', 'Create') 4972
(5253, 'JsonScheduleV1', 'Delete') 2591

 (5251, 'JsonAssociationV1', 'Create')
       987  ['CIF_stp_indicator', 'assoc_days',

We can see that delete objects use only sufficiently many fields to identify the record which needs to be deleted, but creates are complete.

It appears that schedules still fall into the same standard/cancellation dichotomy - we confirm this below:

In [28]:
create_shape_by_stp = Counter()

for sequence, object_name in UPDATE_OBJECTS.items():
    for record in iter_schedule_records(BUCKET_NAME, object_name):
        if "JsonScheduleV1" not in record:
            continue

        schedule = record["JsonScheduleV1"]

        if schedule["transaction_type"] != "Create":
            continue

        segment = schedule["schedule_segment"]

        has_locations = "schedule_location" in segment
        has_new_segment = "new_schedule_segment" in schedule

        create_shape_by_stp[
            (
                schedule["CIF_stp_indicator"],
                has_locations,
                has_new_segment,
            )
        ] += 1

create_shape_by_stp

Counter({('O', True, True): 7512,
         ('N', True, True): 7409,
         ('C', False, False): 4356,
         ('P', True, True): 532})

In [47]:
association_nulls_by_transaction = defaultdict(Counter)
association_counts_by_transaction = Counter()

for sequence, object_name in UPDATE_OBJECTS.items():
    for record in iter_schedule_records(BUCKET_NAME, object_name):
        if "JsonAssociationV1" not in record:
            continue

        association = record["JsonAssociationV1"]
        transaction_type = association["transaction_type"]

        association_counts_by_transaction[transaction_type] += 1

        for key, value in association.items():
            if value is None:
                association_nulls_by_transaction[transaction_type][key] += 1

print("Counts:")
pprint(association_counts_by_transaction)

print("\nNulls:")
for transaction_type, counts in association_nulls_by_transaction.items():
    print(f"\n{transaction_type}:")
    pprint(counts)

Python(95884) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(95903) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(95922) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(95941) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(95960) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(95979) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Counts:
Counter({'Create': 3074, 'Delete': 540})

Nulls:

Delete:
Counter({'base_location_suffix': 511})

Create:
Counter({'assoc_location_suffix': 3074, 'base_location_suffix': 2996})
